# Удаление пустых срезов, срезов с черепом, brain windowing, аугментация

In [2]:
# =========================
# 2. Импорты
# =========================
import os
import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import nibabel as nib

from scipy.ndimage import rotate, shift

warnings.filterwarnings("ignore")

# =========================
# 3. Подключение Google Drive
# =========================

# ==========================================================
# 4. Пути
# ==========================================================
BASE_DIR = Path(r"D:")   # текущая папка, где запущен notebook
DATASET_DIR = BASE_DIR / "Data_11_31_12_31"
RESULTS_DIR = BASE_DIR / "Results_finish"
RESULTS_DIR.mkdir(exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# ==========================================================
# 5. Параметры пайплайна
# ==========================================================
WL = 40
WW = 80

LOWER_WINDOW = WL - WW / 2   # 0
UPPER_WINDOW = WL + WW / 2   # 80

AIR_THRESHOLD = -300
AIR_RATIO_MIN = 0.1

BONE_THRESHOLD = 400
BONE_RATIO_MAX = 0.31

BRIGHTNESS_MIN = -0.1
BRIGHTNESS_MAX = 0.1

CONTRAST_MIN = 0.9
CONTRAST_MAX = 1.1

ROTATION_RAD_MIN = -0.125
ROTATION_RAD_MAX = 0.125
ROTATION_DEG_MIN = math.degrees(ROTATION_RAD_MIN)
ROTATION_DEG_MAX = math.degrees(ROTATION_RAD_MAX)

SHIFT_FACTOR = 0.03

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

OUTPUT_DTYPE = np.float32

# ==========================================================
# 6. Вспомогательные функции
# ==========================================================
def list_nifti_files(dataset_dir: str):
    return sorted(Path(dataset_dir).glob("*.nii.gz"))


def extract_study_name(file_path: Path) -> str:
    name = file_path.name
    if name.endswith(".nii.gz"):
        return name[:-7]
    return file_path.stem


def keep_slice_by_hu(slice_hu: np.ndarray) -> bool:
    ratio_air = np.mean(slice_hu > AIR_THRESHOLD)
    ratio_bone = np.mean(slice_hu > BONE_THRESHOLD)

    if ratio_air < AIR_RATIO_MIN:
        return False
    if ratio_bone > BONE_RATIO_MAX:
        return False
    return True


def apply_brain_window_and_normalize(slice_hu: np.ndarray) -> np.ndarray:
    slice_win = np.clip(slice_hu, LOWER_WINDOW, UPPER_WINDOW)
    slice_norm = (slice_win - LOWER_WINDOW) / (UPPER_WINDOW - LOWER_WINDOW)
    return slice_norm.astype(np.float32)


def sample_patient_augmentation_params():
    """
    Один набор параметров на весь объём пациента.
    """
    return {
        "brightness": np.random.uniform(BRIGHTNESS_MIN, BRIGHTNESS_MAX),
        "contrast": np.random.uniform(CONTRAST_MIN, CONTRAST_MAX),
        "angle_deg": np.random.uniform(ROTATION_DEG_MIN, ROTATION_DEG_MAX),
        "height_factor": np.random.uniform(-SHIFT_FACTOR, SHIFT_FACTOR),
        "width_factor": np.random.uniform(-SHIFT_FACTOR, SHIFT_FACTOR),
    }


def augment_slice(slice_norm: np.ndarray, aug_params: dict) -> np.ndarray:
    """
    Для всех срезов пациента используются одинаковые:
    brightness, contrast, rotation, shift.
    Flip тоже одинаковый, и делается так, как нужно для твоего viewer:
    по горизонтали на экране -> np.flipud(x)
    """
    x = slice_norm.copy().astype(np.float32)

    # 1. Brightness
    b = aug_params["brightness"]
    x = x * (1.0 + b)

    # 2. Contrast
    c = aug_params["contrast"]
    m = float(np.mean(x))
    x = (x - m) * c + m

    # 3. Rotation
    angle_deg = aug_params["angle_deg"]
    x = rotate(
        x,
        angle=angle_deg,
        reshape=False,
        order=1,
        mode="constant",
        cval=0.0,
        prefilter=False
    )

    # 4. Flip по горизонтали в viewer
    x = np.flipud(x)

    # 5. Shift
    h, w = x.shape
    shift_y = aug_params["height_factor"] * h
    shift_x = aug_params["width_factor"] * w

    x = shift(
        x,
        shift=(shift_y, shift_x),
        order=1,
        mode="constant",
        cval=0.0,
        prefilter=False
    )

    # Clip в [0,1]
    x = np.clip(x, 0.0, 1.0)

    return x.astype(np.float32)


def process_volume(volume_hu: np.ndarray):
    processed_slices = []
    augmented_slices = []
    kept_indices = []

    z_dim = volume_hu.shape[2]

    # Один набор параметров на пациента
    aug_params = sample_patient_augmentation_params()

    for z in range(z_dim):
        slice_hu = volume_hu[:, :, z]

        if not keep_slice_by_hu(slice_hu):
            continue

        slice_norm = apply_brain_window_and_normalize(slice_hu)
        slice_aug = augment_slice(slice_norm, aug_params)

        processed_slices.append(slice_norm)
        augmented_slices.append(slice_aug)
        kept_indices.append(z)

    if len(processed_slices) == 0:
        return None, None, kept_indices, aug_params

    processed_volume = np.stack(processed_slices, axis=2).astype(OUTPUT_DTYPE)
    augmented_volume = np.stack(augmented_slices, axis=2).astype(OUTPUT_DTYPE)

    return processed_volume, augmented_volume, kept_indices, aug_params


def save_nifti(volume: np.ndarray, reference_img: nib.Nifti1Image, out_path: str):
    new_header = reference_img.header.copy()
    new_header.set_data_dtype(np.float32)

    new_img = nib.Nifti1Image(
        volume.astype(np.float32),
        affine=reference_img.affine,
        header=new_header
    )
    nib.save(new_img, out_path)


# ==========================================================
# 7. Основной запуск
# ==========================================================
all_nifti_files = list_nifti_files(DATASET_DIR)
selected_files = all_nifti_files.copy()

print(f"Всего .nii.gz в папке Data: {len(selected_files)}")

if len(selected_files) == 0:
    raise RuntimeError("В папке Data не найдено ни одного файла .nii.gz.")

log_rows = []

for idx, file_path in enumerate(selected_files, start=1):
    study_name = extract_study_name(file_path)
    print(f"[{idx}/{len(selected_files)}] Обработка: {study_name}")

    try:
        img = nib.load(str(file_path))
        volume_hu = img.get_fdata().astype(np.float32)

        if volume_hu.ndim != 3:
            print(f"  Пропуск: ожидался 3D volume, получено shape={volume_hu.shape}")
            log_rows.append({
                "study_uid": study_name,
                "status": "skipped_not_3d",
                "original_slices": None,
                "kept_slices": None,
                "brightness": None,
                "contrast": None,
                "angle_deg": None,
                "height_factor": None,
                "width_factor": None,
            })
            continue

        processed_volume, augmented_volume, kept_indices, aug_params = process_volume(volume_hu)

        original_z = volume_hu.shape[2]
        kept_z = len(kept_indices)

        if processed_volume is None:
            print("  Все срезы удалены фильтрацией.")
            log_rows.append({
                "study_uid": study_name,
                "status": "all_slices_removed",
                "original_slices": original_z,
                "kept_slices": 0,
                "brightness": aug_params["brightness"],
                "contrast": aug_params["contrast"],
                "angle_deg": aug_params["angle_deg"],
                "height_factor": aug_params["height_factor"],
                "width_factor": aug_params["width_factor"],
            })
            continue

        processed_out = os.path.join(RESULTS_DIR, f"{study_name}.nii.gz")
        augmented_out = os.path.join(RESULTS_DIR, f"{study_name}_aug.nii.gz")

        save_nifti(processed_volume, img, processed_out)
        save_nifti(augmented_volume, img, augmented_out)

        print(f"  Сохранено: {processed_out}")
        print(f"  Сохранено: {augmented_out}")
        print(f"  Срезов: было {original_z}, осталось {kept_z}")
        print(
            f"  Aug params: "
            f"b={aug_params['brightness']:.4f}, "
            f"c={aug_params['contrast']:.4f}, "
            f"angle={aug_params['angle_deg']:.4f} deg, "
            f"shift_y={aug_params['height_factor']:.4f}, "
            f"shift_x={aug_params['width_factor']:.4f}"
        )

        log_rows.append({
            "study_uid": study_name,
            "status": "ok",
            "original_slices": original_z,
            "kept_slices": kept_z,
            "brightness": aug_params["brightness"],
            "contrast": aug_params["contrast"],
            "angle_deg": aug_params["angle_deg"],
            "height_factor": aug_params["height_factor"],
            "width_factor": aug_params["width_factor"],
        })

    except Exception as e:
        print(f"  Ошибка: {e}")
        log_rows.append({
            "study_uid": study_name,
            "status": f"error: {str(e)}",
            "original_slices": None,
            "kept_slices": None,
            "brightness": None,
            "contrast": None,
            "angle_deg": None,
            "height_factor": None,
            "width_factor": None,
        })

log_df = pd.DataFrame(log_rows)
log_path = os.path.join(RESULTS_DIR, "processing_log.xlsx")
log_df.to_excel(log_path, index=False)

print("\nГотово.")
print(f"Лог сохранён: {log_path}")

Всего .nii.gz в папке Data: 177
[1/177] Обработка: 1.2.643.5.1.13.13.12.2.77.8252.10030009090012081505031408100115
  Сохранено: D:Results_finish\1.2.643.5.1.13.13.12.2.77.8252.10030009090012081505031408100115.nii.gz
  Сохранено: D:Results_finish\1.2.643.5.1.13.13.12.2.77.8252.10030009090012081505031408100115_aug.nii.gz
  Срезов: было 279, осталось 247
  Aug params: b=-0.0251, c=1.0901, angle=3.3231 deg, shift_y=0.0059, shift_x=-0.0206
[2/177] Обработка: 1.2.643.5.1.13.13.12.2.77.8252.10030607050404140205080815071508
  Сохранено: D:Results_finish\1.2.643.5.1.13.13.12.2.77.8252.10030607050404140205080815071508.nii.gz
  Сохранено: D:Results_finish\1.2.643.5.1.13.13.12.2.77.8252.10030607050404140205080815071508_aug.nii.gz
  Срезов: было 400, осталось 373
  Aug params: b=-0.0688, c=0.9116, angle=5.2451 deg, shift_y=0.0061, shift_x=0.0125
[3/177] Обработка: 1.2.643.5.1.13.13.12.2.77.8252.10040503121502130200111106020806
  Сохранено: D:Results_finish\1.2.643.5.1.13.13.12.2.77.8252.10040503121

# Удаление лишних срезов, оконная обработка и номрализация без аугментация

In [38]:
# =========================
# 2. Импорты
# =========================
import os
import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import nibabel as nib

from scipy.ndimage import rotate, shift

warnings.filterwarnings("ignore")

# =========================
# 3. Подключение Google Drive
# =========================

# ==========================================================
# 4. Пути
# ==========================================================
BASE_DIR = Path(r"D:")   # текущая папка, где запущен notebook
DATASET_DIR = BASE_DIR / "Dataset_validation"
RESULTS_DIR = BASE_DIR / "Dataset_validation_norm"
RESULTS_DIR.mkdir(exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# ==========================================================
# 5. Параметры пайплайна
# ==========================================================
WL = 40
WW = 80

LOWER_WINDOW = WL - WW / 2   # 0
UPPER_WINDOW = WL + WW / 2   # 80

AIR_THRESHOLD = -300
AIR_RATIO_MIN = 0.1

BONE_THRESHOLD = 400
BONE_RATIO_MAX = 0.31

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

OUTPUT_DTYPE = np.float32

# ==========================================================
# 6. Вспомогательные функции
# ==========================================================
def list_nifti_files(dataset_dir: str):
    return sorted(Path(dataset_dir).glob("*.nii.gz"))


def extract_study_name(file_path: Path) -> str:
    name = file_path.name
    if name.endswith(".nii.gz"):
        return name[:-7]
    return file_path.stem


def keep_slice_by_hu(slice_hu: np.ndarray) -> bool:
    ratio_air = np.mean(slice_hu > AIR_THRESHOLD)
    ratio_bone = np.mean(slice_hu > BONE_THRESHOLD)

    if ratio_air < AIR_RATIO_MIN:
        return False
    if ratio_bone > BONE_RATIO_MAX:
        return False
    return True


def apply_brain_window_and_normalize(slice_hu: np.ndarray) -> np.ndarray:
    slice_win = np.clip(slice_hu, LOWER_WINDOW, UPPER_WINDOW)
    slice_norm = (slice_win - LOWER_WINDOW) / (UPPER_WINDOW - LOWER_WINDOW)
    return slice_norm.astype(np.float32)


def process_volume(volume_hu: np.ndarray):
    processed_slices = []
    kept_indices = []

    z_dim = volume_hu.shape[2]

    for z in range(z_dim):
        slice_hu = volume_hu[:, :, z]

        if not keep_slice_by_hu(slice_hu):
            continue

        slice_norm = apply_brain_window_and_normalize(slice_hu)

        processed_slices.append(slice_norm)
        kept_indices.append(z)

    if len(processed_slices) == 0:
        return None, kept_indices

    processed_volume = np.stack(processed_slices, axis=2).astype(OUTPUT_DTYPE)

    return processed_volume, kept_indices


def save_nifti(volume: np.ndarray, reference_img: nib.Nifti1Image, out_path: str):
    new_header = reference_img.header.copy()
    new_header.set_data_dtype(np.float32)

    new_img = nib.Nifti1Image(
        volume.astype(np.float32),
        affine=reference_img.affine,
        header=new_header
    )
    nib.save(new_img, out_path)


# ==========================================================
# 7. Основной запуск
# ==========================================================
all_nifti_files = list_nifti_files(DATASET_DIR)
selected_files = all_nifti_files.copy()

print(f"Всего .nii.gz в папке Data: {len(selected_files)}")

if len(selected_files) == 0:
    raise RuntimeError("В папке Data не найдено ни одного файла .nii.gz.")

log_rows = []

for idx, file_path in enumerate(selected_files, start=1):
    study_name = extract_study_name(file_path)
    print(f"[{idx}/{len(selected_files)}] Обработка: {study_name}")

    try:
        img = nib.load(str(file_path))
        volume_hu = img.get_fdata().astype(np.float32)

        if volume_hu.ndim != 3:
            print(f"  Пропуск: ожидался 3D volume, получено shape={volume_hu.shape}")
            log_rows.append({
                "study_uid": study_name,
                "status": "skipped_not_3d",
                "original_slices": None,
                "kept_slices": None,
            })
            continue

        processed_volume, kept_indices = process_volume(volume_hu)

        original_z = volume_hu.shape[2]
        kept_z = len(kept_indices)

        if processed_volume is None:
            print("  Все срезы удалены фильтрацией.")
            log_rows.append({
                "study_uid": study_name,
                "status": "all_slices_removed",
                "original_slices": original_z,
                "kept_slices": 0,
            })
            continue

        processed_out = os.path.join(RESULTS_DIR, f"{study_name}.nii.gz")

        save_nifti(processed_volume, img, processed_out)

        print(f"  Сохранено: {processed_out}")
        print(f"  Срезов: было {original_z}, осталось {kept_z}")

        log_rows.append({
            "study_uid": study_name,
            "status": "ok",
            "original_slices": original_z,
            "kept_slices": kept_z,
        })

    except Exception as e:
        print(f"  Ошибка: {e}")
        log_rows.append({
            "study_uid": study_name,
            "status": f"error: {str(e)}",
            "original_slices": None,
            "kept_slices": None,
        })

log_df = pd.DataFrame(log_rows) 
log_path = os.path.join(RESULTS_DIR, "processing_log.xlsx")
log_df.to_excel(log_path, index=False)

print("\nГотово.")
print(f"Лог сохранён: {log_path}")


Всего .nii.gz в папке Data: 105
[1/105] Обработка: 1.2.643.5.1.13.13.12.2.77.8252.00000209100310101115081103041409
  Сохранено: D:Dataset_validation_norm\1.2.643.5.1.13.13.12.2.77.8252.00000209100310101115081103041409.nii.gz
  Срезов: было 275, осталось 252
[2/105] Обработка: 1.2.643.5.1.13.13.12.2.77.8252.00001312020908070201120206020211
  Сохранено: D:Dataset_validation_norm\1.2.643.5.1.13.13.12.2.77.8252.00001312020908070201120206020211.nii.gz
  Срезов: было 393, осталось 363
[3/105] Обработка: 1.2.643.5.1.13.13.12.2.77.8252.00001510120413110515051009061303
  Сохранено: D:Dataset_validation_norm\1.2.643.5.1.13.13.12.2.77.8252.00001510120413110515051009061303.nii.gz
  Срезов: было 289, осталось 289
[4/105] Обработка: 1.2.643.5.1.13.13.12.2.77.8252.00020812061306021302121402040201
  Сохранено: D:Dataset_validation_norm\1.2.643.5.1.13.13.12.2.77.8252.00020812061306021302121402040201.nii.gz
  Срезов: было 245, осталось 218
[5/105] Обработка: 1.2.643.5.1.13.13.12.2.77.8252.00030209100815